# KGRID interactive data explorer
Select WMUs and a UTC interval, inspect available files, download a bounded sample, and export matching **PNG** and offline **HTML** plots.

Run all cells to display controls. Network access occurs only when you press **Discover / preview** or **Download selected**. **Load local dataset** and **Offline synthetic demo** work without credentials.

Setup: run `uv sync`, then `uv run jupyter lab notebooks/kgrid_explorer.ipynb` from the repository root. Credentials belong in `~/.config/kestrelgrid/credentials.json`, never in notebook cells. See `README.md`.

In [ ]:
from pathlib import Path

from IPython.display import display

from kestrelgrid_tools.explorer import Explorer

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists() and (p / 'config/wmu_locations.json').exists()), None)
if ROOT is None:
    raise RuntimeError('Start this notebook inside the kestrelgrid_analysis checkout')
explorer = Explorer(ROOT)
display(explorer.widget)

## How to use

1. Choose WMUs, latest/explicit interval, products and a transfer budget. Default: WMU **4021**, latest closed archive, **60 seconds**, **100 MiB**.
2. Press **Discover / preview**. Review source names and sizes; large `.raw` archives are range-sampled. Press **Download selected**. Downloads are serial.
3. Adjust local plot bounds and traces, then press **Update plots (local)**. Blank plot bounds show all downloaded data. To see individual waveform cycles, select a short interval (e.g. 0.1–1 second).
4. Zoom, pan and toggle traces in the plots. Press **Export current view** to create matching PNG and standalone HTML files under the displayed dataset folder.

The HTML embeds Plotly.js and the plotted samples, so it works offline. Python/widget controls remain notebook-only. Zooming a reduced min/max overview does not reconstruct omitted samples. Statistics use full-resolution data. PNG export requires Chrome/Chromium for Kaleido.

## Timing, quality and frequency

All selections and plot labels use UTC. Live inspection on 2026-11-05 found that daily filenames correspond to **America/Los_Angeles** partitions; the nominal 2026-11-04 raw archive ends around 2026-11-05 08:00 UTC. DST-aware partition conversion is implemented. Actual recording timestamps are retained in the manifest.

Current Parquet columns are `t` (epoch seconds), `rms`, `theta`, and `gps`, at approximately 2 sps. New RMS/frequency schemas are supported explicitly; unknown schemas raise an error. Raw sampling rate comes from each frame header. GPS-equipped does not imply every frame is GPS-locked.

Frequency from theta is **opt-in**: establish units, reference frequency and derivative sign before enabling it. For radians relative to a rotating reference, `f = f_ref + sign * d(theta)/dt / (2*pi)`. Unwrapping is segment-aware, but at 2 sps deviations beyond ±1 Hz can alias undetectably. Stored frequency is used when present. Do not assume the French sensor uses a 60 Hz phase reference.

This server is bandwidth constrained. The notebook will not mirror it. More than 12 waveform archives per selection is blocked; coordinate larger downloads with the provider. Existing local bundles can be reloaded with no network access.

## Access the selected data in Python

After loading a dataset, use `explorer.dataset.waveforms['4021']`, `explorer.dataset.measurements['4021']`, `explorer.dataset.summary`, and `explorer.dataset.manifest`. Dataframes retain full selected samples; plots may use display reduction.

Optional examples (run manually after loading):
```python
dataset = explorer.dataset
display(dataset.summary)
display(dataset.waveforms['4021'].head())
dataset.waveforms['4021'].to_csv(dataset.directory / 'selected_voltage.csv', index=False)
```

For direct public-package CSV conversion of a frame-aligned raw sample:
```python
from kestrelgrid_tools.interpretation import export_raw_csv
raw_path = next(dataset.directory.glob('*.raw'))
export_raw_csv(raw_path, dataset.directory / 'frame_aligned_voltage.csv')
```
The direct raw export includes full boundary frames; the normalized dataframe reflects the exact requested `[start, end)` interval. Clear notebook outputs before sharing or committing.